# Stitch EXP28 — all-positions graft, Procrustes, donor-side INLP
#
# STANDALONE. Does not depend on any earlier notebook's kernel state.
# Run cells 0-2 once, RESTART KERNEL, then run from cell 3 to the end.
#
# Answers, in order:
#   A. Does grafting EVERY prompt position (not just the last) move full-answer conferral?
#   B. Does an unsupervised, information-preserving (Procrustes) map confer?
#   C. If the answer is erased from the DONOR state before the map sees it, does conferral survive?
#   D. (optional) Does a multi-layer task-map graft rescue the full answer?
#
# Every condition is trained and scored IN THIS RUN against the same bin and the same
# seeds, so nothing here is compared across runs.

In [1]:
# Pinned to match every notebook in the repo that already runs on this pod image.
# Do NOT change this to an unpinned `-U transformers`: the current release needs a newer
# torch than the image ships, is_torch_available() then returns False, and every model
# class silently becomes a DummyObject ("requires the PyTorch library" at load time).
!pip install -q "transformers==4.46.3" accelerate datasets numpy huggingface_hub hf_transfer
!pip uninstall -y torchvision torchaudio
print("deps installed — RESTART THE KERNEL now, then continue from CELL 2")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
deps installed — RESTART THE KERNEL now, then continue from CELL 2


In [2]:
# === CELL 2: imports + RUN CONFIG =============================================
import os, json, math, random, time, gc
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Guard the exact failure mode a bad transformers/torch pairing produces: transformers
# decides at import time whether torch exists, and if it decides no, AutoModelForCausalLM
# is a DummyObject that only raises when you try to LOAD a model, four cells later.
import transformers
from transformers.utils import is_torch_available
assert is_torch_available(), (
    f"transformers {transformers.__version__} does not see torch {torch.__version__}. "
    "Re-run cell 1 (it pins transformers==4.46.3) and RESTART THE KERNEL.")
assert hasattr(AutoModelForCausalLM, "from_pretrained") and \
       type(AutoModelForCausalLM).__name__ != "DummyObject", \
    "AutoModelForCausalLM is a DummyObject — transformers/torch mismatch, see cell 1."
print(f"torch {torch.__version__} | transformers {transformers.__version__} | cuda {torch.cuda.is_available()}")

# ---- PICK THE PAIR -----------------------------------------------------------
# Same-tokenizer pairs only: the all-positions graft writes by position, so the two
# models must tokenize identically. Asserted in CELL 5.
PAIR = "qwen"          # "gemma" | "qwen" | "llama"

_CFG = {
  # name      donor                       recipient                  L_r  L_d
  "gemma": ("google/gemma-2-9b",         "google/gemma-2-2b",        13,  34),
  "qwen":  ("Qwen/Qwen2.5-7B",           "Qwen/Qwen2.5-0.5B",        18,  23),
  "llama": ("meta-llama/Llama-3.1-8B",   "meta-llama/Llama-3.2-1B",   12,  17),
}
MODEL_D, MODEL_R, L_R, L_D = _CFG[PAIR]

# ---- which experiments to run ------------------------------------------------
RUN_ALLPOS    = True    # A. all-positions graft + k-sweep + shuffle control   (the headline)
RUN_PROCRUST  = False    # B. Procrustes / alignment-only baseline              (cheap)
RUN_DONOR_INLP= False    # C. donor-side answer erasure                          (Gemma first)
# D. Llama 10x: does Llama's low conferral come from having ~900 first-token classes
#    on 3k examples rather than from its packed tokenizer? Runs as an EXTRA k=1 arm
#    inside the Llama session -- it does NOT replace the 3k baseline, because the
#    three-pair replication requires all three pairs trained on the same 3k.
RUN_LLAMA_10X = (PAIR == "qwen")
RUN_LLAMA_10X = False
N_TRAIN_10X   = 30000

# E. whole-answer objective (CE over EVERY answer token, not just the first) at k=1 and
#    k=all. This is the ceiling condition: widest channel AND the objective that already
#    doubles full-answer conferral at k=1 in the paper. NEW CODE, never run on a GPU.
#    Leave False for the Qwen/Llama replication runs; turn on for a dedicated session.
RUN_WHOLEANS  = True

# multi-layer graft: deliberately NOT run. The k-sweep in EXP-A already tests the capacity
# axis, and does it better -- 50x the bandwidth at one layer moved nothing.

# ---- sizes -------------------------------------------------------------------
N_TRAIN      = 3000     # Llama 10x run: set to 30000 and leave everything else alone
N_EVAL       = 2000
BATCH        = 16
TASK_EPOCHS  = 2
TASK_SEEDS   = [0, 1]   # 2 seeds for both baseline and all-positions (same seeds, paired)
MAX_NEW      = 6
RIDGE_LAMBDA = 1e3
K_SWEEP      = [1, 2, 4, 8, 16, "all"]   # how many trailing prompt positions to write
ALLPOS_CAP   = None     # cap the all-position training set if host RAM is tight (e.g. 1500)

# NOTE: a second session for the same PAIR (whole-answer only, 10x only, more seeds)
# would overwrite the first file. Append a suffix by hand for those runs, e.g.
#   OUT_JSON = f"exp28_allpos_{PAIR}_wholeans.json"
OUT_JSON = f"exp28_allpos_{PAIR}_L{L_R}.json"
RESULTS  = {"_config": dict(pair=PAIR, donor=MODEL_D, recipient=MODEL_R, L_R=L_R, L_D=L_D,
                            n_train=N_TRAIN, n_eval=N_EVAL, epochs=TASK_EPOCHS,
                            seeds=TASK_SEEDS, k_sweep=[str(k) for k in K_SWEEP])}

def save():
    with open(OUT_JSON, "w") as f: json.dump(RESULTS, f, indent=2)
    print(f"  [saved {OUT_JSON}]")

print(f"PAIR={PAIR}  {MODEL_D} (L{L_D})  ->  {MODEL_R} (L{L_R})   device={DEVICE}")

torch 2.4.1+cu124 | transformers 4.46.3 | cuda True
PAIR=qwen  Qwen/Qwen2.5-7B (L23)  ->  Qwen/Qwen2.5-0.5B (L18)   device=cuda


In [ ]:
# === CELL 3: Hugging Face login (Gemma and Llama are gated) ===================
from huggingface_hub import login
HF_TOKEN = ""      # <-- paste, run, then clear before sharing
login(HF_TOKEN)
print("logged in")

logged in


In [4]:
# === CELL 4: statistics helpers (identical to the main eval notebooks) ========
BOOT_B = 2000
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=BOOT_B, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x)
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

def mcnemar_exact(a, b):
    """Exact two-sided McNemar on paired boolean lists. Fractions, not floats:
    the normal approximation and float binomials both break past n~1000."""
    from fractions import Fraction
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01 = int((~a & b).sum()); n10 = int((a & ~b).sum()); n = n01 + n10
    if n == 0: return dict(n01=0, n10=0, p=1.0)
    k = min(n01, n10)
    tot = Fraction(0)
    for i in range(0, k+1):
        tot += Fraction(math.comb(n, i))
    p = min(1.0, float(2 * tot / Fraction(2)**n))
    return dict(n01=n01, n10=n10, p=p)
print("stats ready")

stats ready


In [5]:
# === CELL 5: shared helpers (last-position graft = the paper's setup) =========
import re as _re
PATCH_POS = -1

def _hid(o):  return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W

FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set(); tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
def _parse_first_int(text):
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- last-position graft (the paper's stitch) --------------------------------
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]: return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

# ---- NEW: multi-position graft ----------------------------------------------
# _seqgraft["vec"]  : [B, L, d]  already mapped into recipient space, left-padded to match
# _seqgraft["mask"] : [B, L] bool, True where the graft should overwrite the residual.
# Fires on prefill only (seq len > 1), exactly like patch_vec_batch, so decoding is untouched.
_seqgraft = {"vec": None, "mask": None}
def patch_seq_batch(_m, _i, o):
    h = _hid(o); vec = _seqgraft["vec"]; msk = _seqgraft["mask"]
    if vec is None or h.shape[1] <= 1: return o
    if h.shape[0] != vec.shape[0] or h.shape[1] != vec.shape[1]: return o
    m3 = msk.unsqueeze(-1).to(h.device)
    h2 = torch.where(m3, vec.to(h.dtype).to(h.device), h)
    return _pack(o, h2)

def encode_with_answer(tok, expr, ans):
    """Full token sequence (prompt + answer) and the index of the first answer token.
    f[:j] is exactly the `ids` the rest of the notebook treats as the prompt."""
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f), j

def build_prompt_mask(prompt_lens, total_lens, L, k):
    """True over the last k positions of the PROMPT span of each left-padded row.
    The answer tokens that follow are never overwritten -- they are the target."""
    m = torch.zeros((len(total_lens), L), dtype=torch.bool)
    for i, (pj, Lf) in enumerate(zip(prompt_lens, total_lens)):
        start = L - Lf                     # first real token of this row
        kk = pj if k == "all" else min(int(k), pj)
        m[i, start+pj-kk : start+pj] = True
    return m

def build_seq_mask(lengths, L, k):
    """True on the last min(k, len_i) REAL positions of each left-padded row.
    k='all' writes every real position."""
    B = len(lengths)
    m = torch.zeros((B, L), dtype=torch.bool)
    for i, Li in enumerate(lengths):
        kk = Li if k == "all" else min(int(k), Li)
        m[i, L-kk:] = True
    return m
print("helpers ready")

helpers ready


In [6]:
# === CELL 6: load both models; assert a shared tokenizer =====================
tokenizer = AutoTokenizer.from_pretrained(MODEL_R)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
model_d = AutoModelForCausalLM.from_pretrained(
    MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_r.config.num_hidden_layers, "recipient layers,",
      model_d.config.num_hidden_layers, "donor layers")

with torch.inference_mode():
    _hl = model_d(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0,-1,:]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), "donor produced NaN/Inf logits"
del _hl

_t1 = AutoTokenizer.from_pretrained(MODEL_R); _t2 = AutoTokenizer.from_pretrained(MODEL_D)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _t1(_probe).input_ids == _t2(_probe).input_ids, (
    "tokenizer mismatch — the all-positions graft writes BY POSITION, so this notebook "
    "requires a same-tokenizer pair. Cross-family pairs cannot run EXP-A.")
del _t1, _t2
D_D = model_d.config.hidden_size; D_R = model_r.config.hidden_size
print(f"tokenizers match | d_donor={D_D} d_recip={D_R}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

loaded: 24 recipient layers, 28 donor layers
tokenizers match | d_donor=3584 d_recip=896


In [7]:
# === CELL 7: problems, states (last-pos AND all-pos), bins ===================
train = gen_arith(tokenizer, N_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

@torch.inference_mode()
def states_last_and_top(model, layer, probs, batch=BATCH):
    acc, top = [], []
    for i in range(0, len(probs), batch):
        ids, m = left_pad([p["ids"] for p in probs[i:i+batch]], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        acc.append(out.hidden_states[layer+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return torch.cat(acc), top

@torch.inference_mode()
def states_allpos(model, layer, probs, batch=BATCH):
    """Per-problem donor residuals at EVERY real position. Returns list of [Li, d]
    float16 CPU tensors (left padding stripped)."""
    out = []
    for i in range(0, len(probs), batch):
        chunk = probs[i:i+batch]
        ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
        hs = model(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                   output_hidden_states=True).hidden_states[layer+1]
        L = ids.shape[1]
        for j, p in enumerate(chunk):
            Li = p["ids"].numel()
            out.append(hs[j, L-Li:, :].half().cpu())
    return out

t0 = time.time()
X9t, _        = states_last_and_top(model_d, L_D, train)
X9e, d_top    = states_last_and_top(model_d, L_D, evalp)
keep = [i for i in range(len(evalp)) if d_top[i] == evalp[i]["tok"]]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  donor solves {len(evalp)}")

X2t, _        = states_last_and_top(model_r, L_R, train)
X2e, r_top    = states_last_and_top(model_r, L_R, evalp)
solvable = [r_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv   = [i for i in range(len(evalp)) if solvable[i]]
print(f"  recipient natively solves {len(solv)}/{len(evalp)} | unsolvable bin n={len(unsolv)}")

_tr_allpos = train if ALLPOS_CAP is None else train[:ALLPOS_CAP]
S9t = states_allpos(model_d, L_D, _tr_allpos)
S9e = states_allpos(model_d, L_D, evalp)
_mb = (sum(t.numel() for t in S9t)+sum(t.numel() for t in S9e))*2/1e9
print(f"  all-position donor states cached: {len(S9t)} train, {len(S9e)} eval ({_mb:.2f} GB fp16 CPU)")
print(f"  [{time.time()-t0:.0f}s]")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
RESULTS["bins"] = dict(n_train=len(train), n_donor_solved=len(evalp),
                       n_unsolvable=len(unsolv), n_solvable=len(solv),
                       recipient_native_rate_on_donor_solved=round(len(solv)/max(1,len(evalp)), 4))
save()

arith: 3000 train, 2000 eval
  donor solves 1681
  recipient natively solves 1113/1681 | unsolvable bin n=568
  all-position donor states cached: 3000 train, 1681 eval (2.47 GB fp16 CPU)
  [69s]
  [saved exp28_allpos_qwen_L18.json]


In [8]:
# === CELL 8: scoring harness (k=1 and k=many share one code path) ============
@torch.inference_mode()
def confer_first(Wb, idxs, k=1, donor_idx=None, Xlast=None, Sseq=None):
    """First-token conferral. k=1 uses the paper's last-position graft.
    donor_idx lets a shuffle control feed problem j's donor state to problem i."""
    W, b = Wb
    src = donor_idx if donor_idx is not None else idxs
    use_seq = (k != 1)
    XL = X9e if Xlast is None else Xlast          # donor last-pos states (EXP-C ablates these)
    SQ = S9e if Sseq  is None else Sseq           # donor all-pos states
    hook = model_r.model.layers[L_R].register_forward_hook(
        patch_seq_batch if use_seq else patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), BATCH):
            sub, ssub = idxs[i:i+BATCH], src[i:i+BATCH]
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            L = ids.shape[1]
            if use_seq:
                lens = [evalp[j]["ids"].numel() for j in sub]
                vec = torch.zeros((len(sub), L, D_R), dtype=torch.float32)
                for r, (j, sj) in enumerate(zip(sub, ssub)):
                    hd = SQ[sj].float()                         # [Lj, d_donor]
                    n = min(hd.shape[0], lens[r])               # right-align, truncate
                    vec[r, L-n:, :] = ((hd[-n:].to(DEVICE)-mu9d) @ W + b).float().cpu()
                _seqgraft["vec"]  = vec.to(DEVICE)
                _seqgraft["mask"] = build_seq_mask(lens, L, k)
            else:
                _seqgraft["vec"] = None
                _graft["vec"] = (XL[ssub].to(DEVICE)-mu9d) @ W + b
            top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[r] == evalp[sub[r]]["tok"] for r in range(len(sub))]
    finally:
        hook.remove(); _graft["vec"] = None
        _seqgraft["vec"] = None; _seqgraft["mask"] = None
    return out

@torch.inference_mode()
def confer_full(Wb, idxs, k=1, donor_idx=None, Xlast=None, Sseq=None):
    """Free-generate the whole number and compare to gold. Graft is prefill-only."""
    W, b = Wb
    src = donor_idx if donor_idx is not None else idxs
    use_seq = (k != 1)
    XL = X9e if Xlast is None else Xlast          # donor last-pos states (EXP-C ablates these)
    SQ = S9e if Sseq  is None else Sseq           # donor all-pos states
    hook = model_r.model.layers[L_R].register_forward_hook(
        patch_seq_batch if use_seq else patch_vec_batch)
    ok = []
    try:
        for i in range(0, len(idxs), BATCH):
            sub, ssub = idxs[i:i+BATCH], src[i:i+BATCH]
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            L = ids.shape[1]
            if use_seq:
                lens = [evalp[j]["ids"].numel() for j in sub]
                vec = torch.zeros((len(sub), L, D_R), dtype=torch.float32)
                for r, (j, sj) in enumerate(zip(sub, ssub)):
                    hd = SQ[sj].float(); n = min(hd.shape[0], lens[r])
                    vec[r, L-n:, :] = ((hd[-n:].to(DEVICE)-mu9d) @ W + b).float().cpu()
                _seqgraft["vec"]  = vec.to(DEVICE)
                _seqgraft["mask"] = build_seq_mask(lens, L, k)
            else:
                _seqgraft["vec"] = None
                _graft["vec"] = (XL[ssub].to(DEVICE)-mu9d) @ W + b
            gen = model_r.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                   max_new_tokens=MAX_NEW, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for j, t in zip(sub, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - evalp[j]["ans"]) < 0.5)
    finally:
        hook.remove(); _graft["vec"] = None
        _seqgraft["vec"] = None; _seqgraft["mask"] = None
    return ok

@torch.inference_mode()
def native_full(idxs):
    ok = []
    for i in range(0, len(idxs), BATCH):
        sub = idxs[i:i+BATCH]
        ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
        gen = model_r.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                               max_new_tokens=MAX_NEW, do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
        txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
        for j, t in zip(sub, txt):
            pred = _parse_first_int(t)
            ok.append(pred is not None and abs(pred - evalp[j]["ans"]) < 0.5)
    return ok

# length-matched derangement: the shuffle partner must have the SAME token length,
# or the position alignment the graft depends on is silently wrong.
def length_matched_shuffle(idxs, seed=0):
    rng = random.Random(seed)
    by_len = {}
    for j in idxs: by_len.setdefault(evalp[j]["ids"].numel(), []).append(j)
    out = {}
    for Lk, group in by_len.items():
        if len(group) < 2:
            out[group[0]] = group[0]; continue
        perm = group[:]
        for _ in range(50):
            rng.shuffle(perm)
            if all(a != b for a, b in zip(group, perm)): break
        out.update(dict(zip(group, perm)))
    n_fixed = sum(1 for j in idxs if out[j] == j)
    return [out[j] for j in idxs], n_fixed
print("scoring harness ready")

scoring harness ready


In [9]:
# === CELL 9: train maps — k=1 baseline and all-positions, SAME seeds =========
def train_map(k, seeds, epochs=TASK_EPOCHS, tag=""):
    """CE on the first answer token. k=1 reproduces the paper's stitch;
    k='all' writes the mapped donor state at every prompt position.
    Returns (maps, ce_log). ce_log is the per-epoch MEAN training CE per seed --
    if k=all lands far above k=1 it is undertrained, and the comparison is unfair."""
    maps, ce_log = [], {}
    model_r.requires_grad_(False)
    use_seq = (k != 1)
    for seed in seeds:
        torch.manual_seed(seed); random.seed(seed)
        W = Wr.clone().to(DEVICE).requires_grad_(True)
        b = mu2.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        hook = model_r.model.layers[L_R].register_forward_hook(
            patch_seq_batch if use_seq else patch_vec_batch)
        idx = list(range(len(_tr_allpos) if use_seq else len(train)))
        ep_ce = []
        try:
            for ep in range(epochs):
                random.Random(seed*100+ep).shuffle(idx)
                _run, _nb = 0.0, 0
                for s in range(0, len(idx), BATCH):
                    sub = idx[s:s+BATCH]
                    ids, m = left_pad([train[j]["ids"] for j in sub], tokenizer.pad_token_id)
                    L = ids.shape[1]
                    if use_seq:
                        lens = [train[j]["ids"].numel() for j in sub]
                        rows = []
                        for r, j in enumerate(sub):
                            hd = S9t[j].float().to(DEVICE)
                            v  = (hd - mu9d) @ W + b                    # [Lj, d_r], differentiable
                            pad = L - v.shape[0]
                            rows.append(F.pad(v, (0,0,pad,0)) if pad > 0 else v[-L:])
                        _seqgraft["vec"]  = torch.stack(rows)
                        _seqgraft["mask"] = build_seq_mask(lens, L, k).to(DEVICE)
                    else:
                        _graft["vec"] = (X9t[sub].to(DEVICE) - mu9d) @ W + b
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train[j]["tok"] for j in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    _run += float(loss.detach()); _nb += 1
                ep_ce.append(round(_run/max(1, _nb), 4))
                print(f"    {tag} seed{seed} ep{ep}: mean CE {ep_ce[-1]:.4f}")
        finally:
            hook.remove(); _graft["vec"] = None
            _seqgraft["vec"] = None; _seqgraft["mask"] = None
        maps.append((W.detach(), b.detach()))
        ce_log[f"seed{seed}"] = ep_ce
    model_r.requires_grad_(True)
    return maps, ce_log

# The k=1 baseline is the reference EXP-A/B/C/D all measure against. EXP-E does not
# use it, so a whole-answer-only session skips ~40 min of training it would discard.
NEED_K1 = RUN_ALLPOS or RUN_PROCRUST or RUN_DONOR_INLP or RUN_LLAMA_10X
maps_k1, ce_k1 = None, None
t0 = time.time()
if NEED_K1:
    print("training k=1 baseline (same-run reference for everything below)...")
    maps_k1, ce_k1 = train_map(1, TASK_SEEDS, tag="k1")
    RESULTS["_training_ce"] = {"_what": ("per-epoch MEAN training cross-entropy. Compare k1 vs kall: "
                                         "if kall plateaus well above k1 it did not converge in "
                                         f"{TASK_EPOCHS} epochs and its conferral is an underestimate. "
                                         "Raise TASK_EPOCHS and rerun cell 9 before believing a null."),
                               "k1": ce_k1}
    print(f"  [{time.time()-t0:.0f}s]")
else:
    RESULTS["_training_ce"] = {"_note": "k=1 baseline not trained this session (EXP-A/B/C/D all off)"}
    print("skipping the k=1 baseline — nothing in this session uses it")

maps_all = None
if RUN_ALLPOS:
    t0 = time.time()
    print("training all-positions map...")
    maps_all, ce_all = train_map("all", TASK_SEEDS, tag="allpos")
    RESULTS["_training_ce"]["kall"] = ce_all
    _f1 = ce_k1[f"seed{TASK_SEEDS[0]}"][-1]; _fa = ce_all[f"seed{TASK_SEEDS[0]}"][-1]
    print(f"  final mean CE: k1={_f1:.4f}  kall={_fa:.4f}  "
          f"{'(kall converged comparably)' if _fa <= _f1*1.25 else '*** kall LOOKS UNDERTRAINED — raise TASK_EPOCHS ***'}")
    print(f"  [{time.time()-t0:.0f}s]")
save()

training k=1 baseline (same-run reference for everything below)...
    k1 seed0 ep0: mean CE 1.1509
    k1 seed0 ep1: mean CE 0.5005
    k1 seed1 ep0: mean CE 1.2608
    k1 seed1 ep1: mean CE 0.5349
  [48s]
training all-positions map...
    allpos seed0 ep0: mean CE 0.9702
    allpos seed0 ep1: mean CE 0.5204
    allpos seed1 ep0: mean CE 0.9685
    allpos seed1 ep1: mean CE 0.5583
  final mean CE: k1=0.5005  kall=0.5204  (kall converged comparably)
  [61s]
  [saved exp28_allpos_qwen_L18.json]


In [10]:
# === CELL 10: EXP-A — all-positions graft, k-sweep, shuffle control ==========
if RUN_ALLPOS:
    A = {"_what": ("Graft the mapped donor state at the last k prompt positions at L_R "
                   "(k=1 is the paper's stitch). The k-sweep uses the all-positions map "
                   "evaluated with a narrower write mask; k=1_trained is the separately "
                   "trained baseline. Shuffle is length-matched so positions stay aligned.")}
    A["native_unsolv_full"]  = fmt(wilson_bools(native_full(unsolv)))

    # the paper's stitch, retrained here so every number below is same-run
    f1 = [confer_first(m, unsolv, k=1) for m in maps_k1]
    F1 = [confer_full(m, unsolv, k=1)  for m in maps_k1]
    A["k1_trained_first"] = fmt(wilson_bools(f1[0]))
    A["k1_trained_full_acrossseed"] = fmt(across_seed_ci([float(np.mean(x)) for x in F1]))
    A["k1_trained_full_seed0"] = fmt(wilson_bools(F1[0]))
    save()

    # all-positions, trained at k=all
    fa = [confer_first(m, unsolv, k="all") for m in maps_all]
    Fa = [confer_full(m, unsolv, k="all")  for m in maps_all]
    A["kall_first"] = fmt(wilson_bools(fa[0]))
    A["kall_full_acrossseed"] = fmt(across_seed_ci([float(np.mean(x)) for x in Fa]))
    A["kall_full_seed0"] = fmt(wilson_bools(Fa[0]))

    # THE headline test: does writing every position move the FULL answer?
    A["mcnemar_full_k1_vs_kall"] = mcnemar_exact(F1[0], Fa[0])
    A["mcnemar_first_k1_vs_kall"] = mcnemar_exact(f1[0], fa[0])
    save()

    # dose-response: same (all-positions) map, progressively narrower write mask
    sweep = {}
    for k in K_SWEEP:
        kf = confer_first(maps_all[0], unsolv, k=k)
        kF = confer_full(maps_all[0], unsolv, k=k)
        sweep[str(k)] = dict(first=fmt(wilson_bools(kf)), full=fmt(wilson_bools(kF)))
        print(f"  k={k:>3}: first {sweep[str(k)]['first']}  full {sweep[str(k)]['full']}")
        A["k_sweep"] = sweep; save()

    # specificity: every position overwritten from a DIFFERENT problem's donor run.
    # Without this, a positive k=all result cannot be told apart from generic prompt translation.
    sh, n_fixed = length_matched_shuffle(unsolv, seed=0)
    A["shuffle_n_fixed_points"] = n_fixed
    A["kall_shuffle_first"] = fmt(wilson_bools(confer_first(maps_all[0], unsolv, k="all", donor_idx=sh)))
    A["kall_shuffle_full"]  = fmt(wilson_bools(confer_full(maps_all[0], unsolv, k="all", donor_idx=sh)))
    A["k1_shuffle_first"]   = fmt(wilson_bools(confer_first(maps_k1[0],  unsolv, k=1,     donor_idx=sh)))

    # does the stitch still break problems the recipient already solved?
    if solv:
        A["kall_solv_first"] = fmt(wilson_bools(confer_first(maps_all[0], solv, k="all")))
        A["k1_solv_first"]   = fmt(wilson_bools(confer_first(maps_k1[0],  solv, k=1)))
    RESULTS["A_all_positions"] = A; save()
    print(json.dumps({k:v for k,v in A.items() if k!="_what"}, indent=2))

  [saved exp28_allpos_qwen_L18.json]
  [saved exp28_allpos_qwen_L18.json]
  k=  1: first 0.731 [0.693, 0.765]  full 0.042 [0.029, 0.062]
  [saved exp28_allpos_qwen_L18.json]
  k=  2: first 0.780 [0.744, 0.812]  full 0.004 [0.001, 0.013]
  [saved exp28_allpos_qwen_L18.json]
  k=  4: first 0.782 [0.746, 0.814]  full 0.002 [0.000, 0.010]
  [saved exp28_allpos_qwen_L18.json]
  k=  8: first 0.780 [0.744, 0.812]  full 0.002 [0.000, 0.010]
  [saved exp28_allpos_qwen_L18.json]
  k= 16: first 0.783 [0.748, 0.815]  full 0.002 [0.000, 0.010]
  [saved exp28_allpos_qwen_L18.json]
  k=all: first 0.789 [0.753, 0.820]  full 0.000 [0.000, 0.007]
  [saved exp28_allpos_qwen_L18.json]
  [saved exp28_allpos_qwen_L18.json]
{
  "native_unsolv_full": "0.000 [0.000, 0.007]",
  "k1_trained_first": "0.789 [0.753, 0.820]",
  "k1_trained_full_acrossseed": "0.041 [0.030, 0.053]",
  "k1_trained_full_seed0": "0.040 [0.027, 0.060]",
  "kall_first": "0.789 [0.753, 0.820]",
  "kall_full_acrossseed": "0.000 [0.000, 0.000

In [11]:
# === CELL 11: EXP-B — Procrustes (unsupervised, information-preserving) ======
if RUN_PROCRUST:
    B = {"_what": ("Orthogonal Procrustes: R = U V^T from SVD(A^T B) on centered paired states. "
                   "R has orthonormal columns (R^T R = I), so it applies NO SHRINKAGE: inside the "
                   "d_recipient-dimensional subspace it keeps, inner products and distances are "
                   "preserved exactly. Ridge does not do this -- it shrinks low-variance donor "
                   "directions toward zero, which is where a low-variance answer signal would "
                   "live. NOT lossless: mapping d_donor -> d_recipient discards a "
                   "(d_donor - d_recipient)-dimensional subspace, so donor states that differ "
                   "ONLY inside that discarded subspace do collapse together. Unsupervised -- "
                   "fit on activations alone, no answer labels. This is the alignment-only "
                   "baseline that the reconstruction map, by objective, cannot be.")}
    A_, B_ = (X9t - X9t.mean(0)).double(), (X2t - X2t.mean(0)).double()
    U, S, Vh = torch.linalg.svd(A_.T @ B_, full_matrices=False)   # [d_d, d_r]
    R = (U @ Vh).float()                                           # semi-orthogonal d_d -> d_r
    B["semiorthogonality_err"] = float((R.T@R - torch.eye(D_R)).abs().max())

    # unscaled, and norm-matched (a semi-orthogonal map lands at donor-scale norms,
    # which the recipient may simply not tolerate -- report both so that is visible)
    src = (X9e - X9t.mean(0)).float()
    s_fit = float((X2e - X2t.mean(0)).norm(dim=1).mean() / (src @ R).norm(dim=1).mean())
    B["norm_match_scale"] = round(s_fit, 4)
    for tag, sc in [("unscaled", 1.0), ("normmatched", s_fit)]:
        Wp = (R * sc).to(DEVICE); bp = mu2.to(DEVICE)
        pm = (Wp, bp)
        B[f"{tag}_unsolv_first"] = fmt(wilson_bools(confer_first(pm, unsolv, k=1)))
        B[f"{tag}_unsolv_full"]  = fmt(wilson_bools(confer_full(pm, unsolv, k=1)))
        rc = F.cosine_similarity(((X9e.to(DEVICE)-mu9d)@Wp+bp).cpu(), X2e, dim=1).numpy()
        B[f"{tag}_recon_cos"] = fmt(bootstrap_ci(rc))
        print(f"  procrustes {tag}: first {B[f'{tag}_unsolv_first']}  full {B[f'{tag}_unsolv_full']}")

    # references measured in THIS run, so nothing is compared across runs
    rm = (Wr.to(DEVICE), mu2.to(DEVICE))
    B["_ref_recon_unsolv_first"] = fmt(wilson_bools(confer_first(rm, unsolv, k=1)))
    B["_ref_recon_unsolv_full"]  = fmt(wilson_bools(confer_full(rm, unsolv, k=1)))
    B["_ref_task_unsolv_first"]  = fmt(wilson_bools(confer_first(maps_k1[0], unsolv, k=1)))
    RESULTS["B_procrustes"] = B; save()
    print(json.dumps({k:v for k,v in B.items() if k!="_what"}, indent=2))

In [12]:
# === CELL 12: EXP-C — donor-side INLP ========================================
# Erase the answer from the DONOR state before the frozen task map ever sees it.
# Nothing about the map's own geometry enters this, so the energy confound that makes
# an output-side SVD ablation uninterpretable does not apply here.
if RUN_DONOR_INLP:
    C_ = {"_what": ("Iteratively null-project the first-digit directions out of the donor "
                    "state h_d, then push the ABLATED donor state through the UNCHANGED "
                    "task map. Control removes the same number of random donor directions. "
                    "The map is frozen throughout; only its input changes.")}
    def _fd(x): return int(str(abs(int(round(x))))[0])
    ytr = (torch.tensor([_fd(p["ans"]) for p in train]) - 1).to(DEVICE)

    def fit_probe(X, y, steps=300, seed=0):
        torch.manual_seed(seed)
        lin = torch.nn.Linear(X.shape[1], 9).to(DEVICE)
        opt = torch.optim.Adam(lin.parameters(), lr=1e-3)
        Xd = X.to(DEVICE)
        for _ in range(steps):
            loss = F.cross_entropy(lin(Xd), y)
            opt.zero_grad(); loss.backward(); opt.step()
        return lin

    EYE  = torch.eye(D_D, device=DEVICE)
    Xtr  = X9t.clone().to(DEVICE)
    Xte0 = X9e.clone().to(DEVICE)
    P    = EYE.clone()
    curve, rank = [], 0
    RANKS = [9, 27, 54, 108, 216, 360, 540]
    for target in RANKS:
        while rank < target:
            lin = fit_probe(Xtr, ytr, seed=rank)
            Q, _ = torch.linalg.qr(lin.weight.detach().T)     # orthonormal basis of the 9 dirs
            Pk = EYE - Q @ Q.T
            P = Pk @ P; Xtr = Xtr @ Pk.T; rank += Q.shape[1]
        acc = float((fit_probe(Xtr, ytr, seed=1000+rank)(Xtr).argmax(1) == ytr).float().mean())
        cf = fmt(wilson_bools(confer_first(maps_k1[0], unsolv, k=1,
                                           Xlast=(Xte0 @ P.T).cpu())))
        g  = torch.linalg.qr(torch.randn(D_D, rank, device=DEVICE,
                 generator=torch.Generator(device=DEVICE).manual_seed(rank)))[0]
        cr = fmt(wilson_bools(confer_first(maps_k1[0], unsolv, k=1,
                                           Xlast=(Xte0 @ (EYE - g @ g.T).T).cpu())))
        curve.append(dict(rank=int(rank), donor_probe_acc_after=round(acc, 4),
                          confer_answer_erased=cf, confer_random_erased=cr))
        print(f"  donor rank {rank}: probe {acc:.3f} | answer-erased {cf} | random {cr}")
        C_["curve"] = curve; save()
    C_["baseline_first"] = fmt(wilson_bools(confer_first(maps_k1[0], unsolv, k=1)))
    C_["donor_probe_acc_unablated"] = round(
        float((fit_probe(X9t.to(DEVICE), ytr)(X9t.to(DEVICE)).argmax(1) == ytr).float().mean()), 4)
    del EYE, Xtr, Xte0, P; gc.collect(); torch.cuda.empty_cache()
    RESULTS["C_donor_inlp"] = C_; save()

In [13]:
# === CELL 13: EXP-D — Llama 10x data (extra k=1 arm, Llama only) =============
# Trained and scored in THIS session against the SAME unsolvable bin as the 3k
# baseline, so the two are directly paired. It does not touch maps_k1.
if RUN_LLAMA_10X:
    t0 = time.time()
    print(f"EXP-D: retraining the k=1 map on {N_TRAIN_10X} problems...")
    train10 = gen_arith(tokenizer, N_TRAIN_10X, random.Random(7), {p["expr"] for p in evalp})
    print(f"  generated {len(train10)} problems (excludes the eval set)")
    X9t10, _ = states_last_and_top(model_d, L_D, train10)
    X2t10, _ = states_last_and_top(model_r, L_R, train10)
    mu9_10, mu2_10, Wr10 = fit_ridge(X9t10, X2t10)
    mu9d10 = mu9_10.to(DEVICE)

    maps10, ce10 = [], {}
    model_r.requires_grad_(False)
    for seed in TASK_SEEDS:
        torch.manual_seed(seed); random.seed(seed)
        W = Wr10.clone().to(DEVICE).requires_grad_(True)
        b = mu2_10.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        hook = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch)
        idx = list(range(len(train10))); ep_ce = []
        try:
            for ep in range(TASK_EPOCHS):
                random.Random(seed*100+ep).shuffle(idx); _run, _nb = 0.0, 0
                for s_ in range(0, len(idx), BATCH):
                    sub = idx[s_:s_+BATCH]
                    _graft["vec"] = (X9t10[sub].to(DEVICE) - mu9d10) @ W + b
                    ids, m = left_pad([train10[j]["ids"] for j in sub], tokenizer.pad_token_id)
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([train10[j]["tok"] for j in sub], device=DEVICE)
                    loss = F.cross_entropy(lg, tgt)
                    opt.zero_grad(); loss.backward(); opt.step()
                    _run += float(loss.detach()); _nb += 1
                ep_ce.append(round(_run/max(1, _nb), 4))
                print(f"    10x seed{seed} ep{ep}: mean CE {ep_ce[-1]:.4f}")
        finally:
            hook.remove(); _graft["vec"] = None
        maps10.append((W.detach(), b.detach())); ce10[f"seed{seed}"] = ep_ce
    model_r.requires_grad_(True)

    # the 10x map reads donor states centred on ITS OWN training mean
    _mu_bk = mu9d; globals()["mu9d"] = mu9d10
    f10 = confer_first(maps10[0], unsolv, k=1)
    F10 = [confer_full(m, unsolv, k=1) for m in maps10]
    globals()["mu9d"] = _mu_bk

    D = {"_what": (f"k=1 task map trained on {len(train10)} problems instead of {len(train)}, "
                   "scored on the same unsolvable bin as the 3k baseline in EXP-A. Tests whether "
                   "Llama's low conferral is a data/class-count problem rather than a tokenizer one."),
         "n_train_10x": len(train10),
         "ce_10x": ce10,
         "first_10x": fmt(wilson_bools(f10)),
         "full_10x_seed0": fmt(wilson_bools(F10[0])),
         "full_10x_acrossseed": fmt(across_seed_ci([float(np.mean(x)) for x in F10])),
         "_ref_first_3k": RESULTS.get("A_all_positions", {}).get("k1_trained_first"),
         "_ref_full_3k_seed0": RESULTS.get("A_all_positions", {}).get("k1_trained_full_seed0"),
         "mcnemar_first_3k_vs_10x": mcnemar_exact(confer_first(maps_k1[0], unsolv, k=1), f10)}
    RESULTS["D_data_10x"] = D; save()
    del X9t10, X2t10, train10; gc.collect(); torch.cuda.empty_cache()
    print(json.dumps({k: v for k, v in D.items() if k != "_what"}, indent=2))
    print(f"  [{time.time()-t0:.0f}s]")

In [14]:
# === CELL 14: EXP-E — whole-answer objective at k=1 and k=all ================
# The paper's maps are trained on the FIRST answer token. Its own Table 1 shows that
# training on the whole answer takes full-answer conferral from 0.13 to 0.25 at k=1.
# This arm asks the ceiling question: widest channel AND best objective, together.
# NEW CODE -- logic dry-run on stubs only, never executed on a GPU. Read the CE before
# trusting any conferral number it produces.
if RUN_WHOLEANS:
    t0 = time.time()
    seqs = {}
    for name, probs in (("train", train), ("eval", evalp)):
        acc = []
        for p in probs:
            full, j = encode_with_answer(tokenizer, p["expr"], p["ans"])
            acc.append(None if full is None else (full, j))
        seqs[name] = acc
    # S9t only covers _tr_allpos, which ALLPOS_CAP can truncate — never index past it
    ok_tr = [i for i, v in enumerate(seqs["train"]) if v is not None and i < len(S9t)]
    ok_ev = [i for i, v in enumerate(seqs["eval"])  if v is not None]
    print(f"EXP-E: {len(ok_tr)}/{len(train)} train and {len(ok_ev)}/{len(evalp)} eval sequences encode cleanly")

    def train_wholeanswer(k, seed):
        torch.manual_seed(seed); random.seed(seed)
        W = Wr.clone().to(DEVICE).requires_grad_(True)
        b = mu2.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        hook = model_r.model.layers[L_R].register_forward_hook(patch_seq_batch)
        idx = ok_tr[:]; ep_ce = []
        try:
            for ep in range(TASK_EPOCHS):
                random.Random(seed*100+ep).shuffle(idx); _run, _nb = 0.0, 0
                for s_ in range(0, len(idx), BATCH):
                    sub = idx[s_:s_+BATCH]
                    fulls = [seqs["train"][j][0] for j in sub]
                    pjs   = [seqs["train"][j][1] for j in sub]
                    ids, m = left_pad(fulls, tokenizer.pad_token_id); L = ids.shape[1]
                    tots = [f.numel() for f in fulls]
                    rows = []
                    for r, j in enumerate(sub):
                        hd = S9t[j].float().to(DEVICE)              # donor states over the PROMPT span
                        v  = (hd - mu9d) @ W + b
                        start = L - tots[r]
                        row = torch.zeros(L, D_R, device=DEVICE, dtype=v.dtype)
                        n = min(v.shape[0], pjs[r])
                        row[start+pjs[r]-n : start+pjs[r]] = v[-n:]
                        rows.append(row)
                    _seqgraft["vec"]  = torch.stack(rows)
                    _seqgraft["mask"] = build_prompt_mask(pjs, tots, L, k).to(DEVICE)
                    lg = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits.float()
                    # predict token i+1 from position i: answer tokens live at [pj, tot)
                    losses = []
                    for r in range(len(sub)):
                        start = L - tots[r]
                        lo, hi = start + pjs[r] - 1, start + tots[r] - 1
                        losses.append(F.cross_entropy(lg[r, lo:hi, :],
                                                      ids[r, lo+1:hi+1].to(DEVICE)))
                    loss = torch.stack(losses).mean()
                    opt.zero_grad(); loss.backward(); opt.step()
                    _run += float(loss.detach()); _nb += 1
                ep_ce.append(round(_run/max(1, _nb), 4))
                print(f"    wholeans k={k} seed{seed} ep{ep}: mean CE {ep_ce[-1]:.4f}")
        finally:
            hook.remove(); _seqgraft["vec"] = None; _seqgraft["mask"] = None
        return (W.detach(), b.detach()), ep_ce

    E = {"_what": ("Maps trained with cross-entropy over EVERY answer token rather than only the "
                   "first, at k=1 and k=all. Scored with the same confer_first/confer_full as "
                   "everything else, so these numbers sit beside EXP-A directly."),
         "n_train_encoded": len(ok_tr)}
    model_r.requires_grad_(False)
    for k in [1, "all"]:
        mp, ce = train_wholeanswer(k, TASK_SEEDS[0])
        E[f"k{k}_ce"] = ce
        E[f"k{k}_first"] = fmt(wilson_bools(confer_first(mp, unsolv, k=k)))
        E[f"k{k}_full"]  = fmt(wilson_bools(confer_full(mp, unsolv, k=k)))
        print(f"  whole-answer k={k}: first {E[f'k{k}_first']}  full {E[f'k{k}_full']}")
        RESULTS["E_whole_answer"] = E; save()
    model_r.requires_grad_(True)
    _A = RESULTS.get("A_all_positions", {})
    E["_ref_firsttok_k1_full"]   = _A.get("k1_trained_full_seed0",
        "not measured this session — compare against the k1_trained_full_seed0 in your "
        "earlier run for this pair (bins are seed-fixed, so they are the same problems)")
    E["_ref_firsttok_kall_full"] = _A.get("kall_full_seed0",
        "not measured this session — see kall_full_seed0 in your earlier run for this pair")
    RESULTS["E_whole_answer"] = E; save()
    print(f"  [{time.time()-t0:.0f}s]")

# === CELL 15: summary ========================================================
save()
print(json.dumps(RESULTS, indent=2)[:6000])
print(f"\n=== wrote {OUT_JSON} ===")
if RUN_ALLPOS:
    A = RESULTS.get("A_all_positions", {})
    print("\nHEADLINE — does writing every position move the full answer?")
    print(f"  k=1   full: {A.get('k1_trained_full_acrossseed')}")
    print(f"  k=all full: {A.get('kall_full_acrossseed')}")
    print(f"  McNemar:    {A.get('mcnemar_full_k1_vs_kall')}")
    print(f"  shuffle (k=all) first: {A.get('kall_shuffle_first')}  <- must stay low")
    ce = RESULTS.get("_training_ce", {})
    print(f"\n  training CE  k1: {ce.get('k1')}")
    print(f"  training CE kall: {ce.get('kall')}")
    print("  (a null at k=all is only meaningful if these plateau at comparable values)")
if RUN_LLAMA_10X and "D_data_10x" in RESULTS:
    D = RESULTS["D_data_10x"]
    print(f"\n10x DATA (Llama):  3k first {D.get('_ref_first_3k')}  ->  30k first {D.get('first_10x')}")
    print(f"                   3k full  {D.get('_ref_full_3k_seed0')}  ->  30k full  {D.get('full_10x_seed0')}")
if RUN_WHOLEANS and "E_whole_answer" in RESULTS:
    E = RESULTS["E_whole_answer"]
    print(f"\nWHOLE-ANSWER OBJECTIVE:  k=1 full {E.get('k1_full')}   k=all full {E.get('kall_full')}")

EXP-E: 3000/3000 train and 1681/1681 eval sequences encode cleanly
    wholeans k=1 seed0 ep0: mean CE 1.7527
    wholeans k=1 seed0 ep1: mean CE 1.3786
  whole-answer k=1: first 0.794 [0.759, 0.825]  full 0.067 [0.049, 0.090]
  [saved exp28_allpos_qwen_L18.json]
    wholeans k=all seed0 ep0: mean CE 1.7242
    wholeans k=all seed0 ep1: mean CE 1.3698
  whole-answer k=all: first 0.801 [0.766, 0.832]  full 0.060 [0.043, 0.082]
  [saved exp28_allpos_qwen_L18.json]
  [saved exp28_allpos_qwen_L18.json]
  [134s]
  [saved exp28_allpos_qwen_L18.json]
{
  "_config": {
    "pair": "qwen",
    "donor": "Qwen/Qwen2.5-7B",
    "recipient": "Qwen/Qwen2.5-0.5B",
    "L_R": 18,
    "L_D": 23,
    "n_train": 3000,
    "n_eval": 2000,
    "epochs": 2,
    "seeds": [
      0,
      1
    ],
    "k_sweep": [
      "1",
      "2",
      "4",
      "8",
      "16",
      "all"
    ]
  },
  "bins": {
    "n_train": 3000,
    "n_donor_solved": 1681,
    "n_unsolvable": 568,
    "n_solvable": 1113,
    "recip

In [15]:
# === DIAGNOSTIC CELL — run AFTER the notebook finishes, same kernel ==========
# Writes its own file: exp28_diag_<PAIR>.json. Touches nothing else.
DIAG_JSON = f"exp28_diag_{PAIR}_L{L_R}.json"
DIAG = {"_what": ("Looks at WHAT the recipient writes, not just whether it is right. "
                  "A full-answer score of 0.000 has two incompatible explanations: the "
                  "model is healthy but does not know the later digits (a result), or the "
                  "graft broke its ability to emit a number at all (a bug). 'right_digit_count' "
                  "separates them -- near 1.0 means healthy, near 0.0 means broken."),
        "pair": PAIR, "n_bin": len(unsolv)}

@torch.inference_mode()
def _gen_texts(Wb, idxs, k=None):
    """Raw continuation per problem. k=None -> no graft (native recipient)."""
    hook = None
    if k is not None:
        hook = model_r.model.layers[L_R].register_forward_hook(
            patch_seq_batch if k != 1 else patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), BATCH):
            sub = idxs[i:i+BATCH]
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            L = ids.shape[1]
            if k is not None:
                W, b = Wb
                if k != 1:
                    lens = [evalp[j]["ids"].numel() for j in sub]
                    vec = torch.zeros((len(sub), L, D_R), dtype=torch.float32)
                    for r, j in enumerate(sub):
                        hd = S9e[j].float(); n = min(hd.shape[0], lens[r])
                        vec[r, L-n:, :] = ((hd[-n:].to(DEVICE) - mu9d) @ W + b).float().cpu()
                    _seqgraft["vec"] = vec.to(DEVICE)
                    _seqgraft["mask"] = build_seq_mask(lens, L, k)
                else:
                    _graft["vec"] = (X9e[sub].to(DEVICE) - mu9d) @ W + b
            gen = model_r.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                   max_new_tokens=MAX_NEW, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
            out += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
    finally:
        if hook: hook.remove()
        _graft["vec"] = None; _seqgraft["vec"] = None; _seqgraft["mask"] = None
    return out

def _integrity(texts, idxs):
    gold = [evalp[j]["ans"] for j in idxs]
    parsed = [_parse_first_int(t) for t in texts]
    got, nd, fd, fu = [], [], [], []
    for p, g in zip(parsed, gold):
        got.append(p is not None)
        if p is None:
            nd.append(False); fd.append(False); fu.append(False); continue
        ps, gs = str(abs(int(p))), str(abs(int(g)))
        nd.append(len(ps) == len(gs)); fd.append(ps[0] == gs[0]); fu.append(abs(p - g) < 0.5)
    return dict(emitted_a_number=fmt(wilson_bools(got)),
                right_digit_count=fmt(wilson_bools(nd)),
                first_digit_correct=fmt(wilson_bools(fd)),
                full_answer_correct=fmt(wilson_bools(fu)),
                mean_chars_emitted=round(float(np.mean([len(t.strip()) for t in texts])), 2),
                examples=[t.strip()[:24] for t in texts[:10]],
                gold_examples=[str(evalp[j]["ans"]) for j in idxs[:10]])

conds = [("native", None, None), ("k1", maps_k1[0] if maps_k1 else None, 1)]
if maps_all: conds.append(("kall", maps_all[0], "all"))
for name, Wb, k in conds:
    if k is not None and Wb is None: continue
    print(f"  generating: {name} ...")
    DIAG[name] = _integrity(_gen_texts(Wb, unsolv, k=k), unsolv)
    print(f"    emitted a number {DIAG[name]['emitted_a_number']} | "
          f"right digit count {DIAG[name]['right_digit_count']} | "
          f"full {DIAG[name]['full_answer_correct']}")
    print(f"    wrote: {DIAG[name]['examples'][:5]}  gold: {DIAG[name]['gold_examples'][:5]}")
    with open(DIAG_JSON, "w") as f: json.dump(DIAG, f, indent=2)

# ---- scaled Procrustes with the least-squares optimal scalar ----------------
A_ = (X9t - X9t.mean(0)).double(); B_ = (X2t - X2t.mean(0)).double()
_U, _S, _Vh = torch.linalg.svd(A_.T @ B_, full_matrices=False)
R = (_U @ _Vh).float()
AR = A_ @ R.double()
s_opt = float((AR * B_).sum() / (AR * AR).sum())     # exact LS scalar, no formula guessing
P = {"optimal_scale": round(s_opt, 6),
     "donor_over_recipient_norm_ratio": round(float(
         (X9e - X9t.mean(0)).norm(dim=1).mean() / (X2e - X2t.mean(0)).norm(dim=1).mean()), 2)}
pm = ((R * s_opt).to(DEVICE), mu2.to(DEVICE))
P["optscale_unsolv_first"] = fmt(wilson_bools(confer_first(pm, unsolv, k=1)))
P["optscale_unsolv_full"]  = fmt(wilson_bools(confer_full(pm, unsolv, k=1)))
DIAG["procrustes_optimal_scale"] = P
with open(DIAG_JSON, "w") as f: json.dump(DIAG, f, indent=2)
print(f"\n  procrustes, optimal scale {s_opt:.4f}: "
      f"first {P['optscale_unsolv_first']}  full {P['optscale_unsolv_full']}")
print(f"\n=== wrote {DIAG_JSON} ===")
print("READ THIS: compare 'right_digit_count' for kall against native and k1.")
print("  near native  -> generation is healthy, a 0.000 full-answer score is a real result")
print("  near zero    -> the graft broke generation, and that 0.000 means nothing")

  generating: native ...
    emitted a number 1.000 [0.993, 1.000] | right digit count 0.731 [0.693, 0.765] | full 0.000 [0.000, 0.007]
    wrote: ['399\n10', '71\n10 *', '395\n10', '196\n10', '795\n10']  gold: ['403', '69', '445', '276', '837']
  generating: k1 ...
    emitted a number 1.000 [0.993, 1.000] | right digit count 0.894 [0.866, 0.917] | full 0.040 [0.027, 0.060]
    wrote: ['343\n10', '69\n10 *', '455\n12', '296\n10', '895\n12']  gold: ['403', '69', '445', '276', '837']
  generating: kall ...
    emitted a number 1.000 [0.993, 1.000] | right digit count 0.000 [0.000, 0.007] | full 0.000 [0.000, 0.007]
    wrote: ['411111', '611111', '411111', '211211', '811111']  gold: ['403', '69', '445', '276', '837']

  procrustes, optimal scale 0.1267: first 0.410 [0.370, 0.451]  full 0.016 [0.008, 0.030]

=== wrote exp28_diag_qwen_L18.json ===
READ THIS: compare 'right_digit_count' for kall against native and k1.
  near native  -> generation is healthy, a 0.000 full-answer score is a 

In [ ]:
# === BLEND / NORM-MATCH SWEEP (FIXED) — run after the notebook, same kernel ==
# FIX vs the previous version: norm-matching is applied ONLY to the context
# positions. The final position keeps the RAW graft, so the paper's stitch is
# untouched and first-token conferral stays comparable across every row.
# Requires _integrity() from the diagnostic cell.
BLEND_JSON = f"exp28_blend_{PAIR}_L{L_R}.json"
BLEND = {"_what": ("alpha blends the CONTEXT positions toward the recipient's own state; "
                   "the final position is always fully grafted with the RAW (un-rescaled) "
                   "vector. normmatch rescales only the context grafts to the norm the "
                   "recipient natively had at that position -- direction untouched, cosine 1.0. "
                   "alpha=0 reproduces the k=1 stitch; alpha=1 without normmatch reproduces "
                   "plain k=all. Any row whose right_digit_count is far below native is "
                   "measuring generation collapse, not conferral."),
         "pair": PAIR, "L_R": L_R, "L_D": L_D, "n_bin": len(unsolv),
         "alphas": [0.25, 0.5, 0.75, 1.0]}

_blend = {"vec": None, "amap": None, "normmatch": False, "lastmask": None}
def patch_seq_blend(_m, _i, o):
    h = _hid(o); vec = _blend["vec"]; amap = _blend["amap"]
    if vec is None or h.shape[1] <= 1: return o
    if h.shape[0] != vec.shape[0] or h.shape[1] != vec.shape[1]: return o
    v = vec.to(h.dtype).to(h.device)
    if _blend["normmatch"]:
        vn = v * (h.norm(dim=-1, keepdim=True) / v.norm(dim=-1, keepdim=True).clamp_min(1e-6))
        lm = _blend["lastmask"].unsqueeze(-1).to(h.device)
        v  = torch.where(lm, v, vn)          # <-- FIX: final position keeps the raw graft
    a = amap.to(h.dtype).to(h.device).unsqueeze(-1)
    return _pack(o, (1 - a) * h + a * v)

def _alpha_map(lengths, L, alpha):
    A = torch.zeros((len(lengths), L))
    for i, Li in enumerate(lengths):
        A[i, L - Li:] = alpha
        A[i, L - 1] = 1.0                    # final position always fully grafted
    return A

@torch.inference_mode()
def _blend_run(Wb, idxs, alpha, normmatch, generate=True):
    W, b = Wb; _blend["normmatch"] = normmatch
    hook = model_r.model.layers[L_R].register_forward_hook(patch_seq_blend)
    texts, firsts = [], []
    try:
        for i in range(0, len(idxs), BATCH):
            sub = idxs[i:i+BATCH]
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            L = ids.shape[1]; lens = [evalp[j]["ids"].numel() for j in sub]
            vec = torch.zeros((len(sub), L, D_R), dtype=torch.float32)
            for r, j in enumerate(sub):
                hd = S9e[j].float(); n = min(hd.shape[0], lens[r])
                vec[r, L-n:, :] = ((hd[-n:].to(DEVICE) - mu9d) @ W + b).float().cpu()
            _blend["vec"] = vec.to(DEVICE)
            _blend["amap"] = _alpha_map(lens, L, alpha)
            _lm = torch.zeros((len(sub), L), dtype=torch.bool); _lm[:, L-1] = True
            _blend["lastmask"] = _lm
            if generate:
                gen = model_r.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                       max_new_tokens=MAX_NEW, do_sample=False,
                                       pad_token_id=tokenizer.eos_token_id)
                texts += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            else:
                top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                firsts += [top[r] == evalp[sub[r]]["tok"] for r in range(len(sub))]
    finally:
        hook.remove()
        _blend["vec"] = None; _blend["amap"] = None
        _blend["normmatch"] = False; _blend["lastmask"] = None
    return texts if generate else firsts

for nm in [False, True]:
    for a in BLEND["alphas"]:
        tag = f"alpha{a}" + ("_normmatch" if nm else "")
        first = _blend_run(maps_all[0], unsolv, a, nm, generate=False)
        r = _integrity(_blend_run(maps_all[0], unsolv, a, nm, generate=True), unsolv)
        r["first_token_conferral"] = fmt(wilson_bools(first))
        BLEND[tag] = r
        print(f"  {tag:22s} first {r['first_token_conferral']:22s} "
              f"digits {r['right_digit_count']:22s} full {r['full_answer_correct']}")
        print(f"    wrote: {r['examples'][:4]}")
        with open(BLEND_JSON, "w") as f: json.dump(BLEND, f, indent=2)

print(f"\n=== wrote {BLEND_JSON} ===")
print("FIRST CHECK: first_token_conferral should now be roughly CONSTANT across all")
print("eight rows. If the normmatch rows still drop, the final-position fix did not take.")
print("THEN: does any row hold right_digit_count near native AND beat k=1 on full answer?")

  alpha0.25              first 0.794 [0.759, 0.825]   digits 0.000 [0.000, 0.007]   full 0.000 [0.000, 0.007]
    wrote: ['411100', '610000', '415010', '210010']
  alpha0.5               first 0.792 [0.757, 0.824]   digits 0.000 [0.000, 0.007]   full 0.000 [0.000, 0.007]
    wrote: ['411111', '611001', '411111', '211210']
  alpha0.75              first 0.787 [0.751, 0.819]   digits 0.000 [0.000, 0.007]   full 0.000 [0.000, 0.007]
    wrote: ['411111', '611011', '411111', '211210']
  alpha1.0               first 0.789 [0.753, 0.820]   digits 0.000 [0.000, 0.007]   full 0.000 [0.000, 0.007]
    wrote: ['411111', '611111', '411111', '211211']
  alpha0.25_normmatch    first 0.782 [0.746, 0.814]   digits 0.815 [0.781, 0.845]   full 0.039 [0.026, 0.058]
    wrote: ['401\n10', '69\n10 *', '405\n10', '288\n10']
